# Step 2 — Two-stage model, stricter tests, real candidate compounds

**Click Runtime ▸ Run all.** Nothing to upload. Expect 20–40 minutes on free Colab (the Materials Project download is the slow part).

What changed from Step 1:
1. **Two-stage model:** first decide metal vs non-metal, then predict the gap only for non-metals.
2. **Stricter tests:** random folds, chemical-system folds, and *leave-one-anion-family-out* (e.g. no oxides in training).
3. **Real compounds:** candidates come from Materials Project formulas with negative formation energy, not invented formulas.
4. **Better ranking:** by the estimated *probability of landing in the 1.0–1.8 eV window*, not closeness to 1.4 eV.

If any cell turns red, **stop and send me the last 5 lines.**

## Cell 1 — Install
If Colab shows *Restart session*, click it, then Run all again.

In [ ]:
%pip install -q matminer pymatgen scikit-learn pandas numpy scipy matplotlib

## Cell 2 — Load experimental data, build features

In [ ]:
import os, re, math, json, itertools, shutil, warnings
from collections import Counter
from functools import reduce
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

SEED = 42
OUT = "step2_outputs"
os.makedirs(OUT, exist_ok=True)

from matminer.datasets import load_dataset
raw = load_dataset("matbench_expt_gap")
target_col = [c for c in raw.columns if "gap" in str(c).lower()][0]
formula_col = [c for c in raw.columns if c != target_col][0]
df = pd.DataFrame({"formula": raw[formula_col].astype(str), "gap": raw[target_col].astype(float)})
print("Rows:", len(df), "(expected 4604)")
assert len(df) > 4000, "Dataset looks wrong - stop and send me this message."

TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)")
def parse_formula(f):
    comp = {}
    for el, n in TOKEN.findall(f):
        comp[el] = comp.get(el, 0.0) + (float(n) if n else 1.0)
    return comp
def chem_system(f): return "-".join(sorted(parse_formula(f)))
def frac_key(f):
    c = parse_formula(f); t = sum(c.values())
    return tuple(sorted((e, round(n / t, 5)) for e, n in c.items()))

CHALC, HAL, PNICT = {"S","Se","Te"}, {"F","Cl","Br","I"}, {"N","P","As","Sb"}
def family(f):
    els = set(parse_formula(f))
    if "O" in els: return "oxide"
    if els & CHALC: return "chalcogenide"
    if els & HAL: return "halide"
    if els & PNICT: return "pnictide"
    return "other"

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
try:
    EP = ElementProperty.from_preset("magpie", impute_nan=True)
except TypeError:
    EP = ElementProperty.from_preset("magpie")
def featurize(formulas):
    rows = [EP.featurize(Composition(parse_formula(f))) for f in formulas]
    return np.nan_to_num(np.array(rows, dtype=float))

ok = df.formula.apply(lambda f: len(parse_formula(f)) > 0)
print("Unparseable rows dropped:", int((~ok).sum()))
df = df[ok].reset_index(drop=True)
df["system"] = df.formula.apply(chem_system)
df["family"] = df.formula.apply(family)
df["key"] = df.formula.apply(frac_key)
y = df.gap.values
nm = (y > 0).astype(int)          # 1 = non-metal (gap > 0), 0 = metal (gap == 0)
print("Computing chemical features (1-2 minutes)...")
X = featurize(df.formula.tolist())
print("Features:", X.shape, "| non-metal share: %.2f" % nm.mean())
print(df.family.value_counts().to_dict())

## Cell 3 — Three test protocols, single-stage vs two-stage model
- **random**: ordinary shuffled 5-fold (easy)
- **system**: whole chemical systems held out (Step 1 test)
- **family**: one whole anion family held out at a time (hard stress test)

In [ ]:
N_FOLDS = 5
def random_folds(n):
    return list(KFold(N_FOLDS, shuffle=True, random_state=18012019).split(np.arange(n)))
def grouped_folds(groups, seed=SEED):
    rng = np.random.default_rng(seed)
    cnt = Counter(groups); keys = list(cnt); rng.shuffle(keys); keys.sort(key=lambda k: -cnt[k])
    load = np.zeros(N_FOLDS); assign = {}
    for k in keys:
        j = int(np.argmin(load)); assign[k] = j; load[j] += cnt[k]
    fo = np.array([assign[g] for g in groups])
    return [(np.where(fo != j)[0], np.where(fo == j)[0]) for j in range(N_FOLDS)]
def family_folds(fams):
    # hold out one anion family at a time; "other" (mostly metals/intermetallics) always stays in training
    return [(np.where(fams != g)[0], np.where(fams == g)[0]) for g in sorted(set(fams)) if g != "other"]

def rf_reg(n=200): return RandomForestRegressor(n_estimators=n, n_jobs=-1, random_state=SEED)
def rf_clf(n=200): return RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=SEED)

def p_nonmetal(clf, Xq):
    return clf.predict_proba(Xq)[:, list(clf.classes_).index(1)] if 1 in clf.classes_ else np.zeros(len(Xq))

protocols = {"random": random_folds(len(df)),
             "system": grouped_folds(df.system.values),
             "family": family_folds(df.family.values)}
OOF = {}
for proto, folds in protocols.items():
    single = np.zeros(len(df)); regp = np.zeros(len(df)); pnm = np.zeros(len(df))
    for tr, te in folds:
        if proto == "system":
            assert not (set(df.system.values[tr]) & set(df.system.values[te])), "leak"
        if proto == "family":
            assert not (set(df.family.values[tr]) & set(df.family.values[te])), "leak"
        single[te] = rf_reg().fit(X[tr], y[tr]).predict(X[te])
        trn = tr[nm[tr] == 1]
        regp[te] = rf_reg().fit(X[trn], y[trn]).predict(X[te])
        pnm[te] = p_nonmetal(rf_clf().fit(X[tr], nm[tr]), X[te])
    tested = np.zeros(len(df), dtype=bool)
    for _, te in folds: tested[te] = True
    OOF[proto] = dict(single=single, reg=regp, pnm=pnm, two=np.where(pnm > 0.5, regp, 0.0), tested=tested)
    print("finished protocol:", proto, flush=True)

rows = []
for proto, o in OOF.items():
    tt = o["tested"]; m1 = (nm == 1) & tt
    rows.append(dict(
        protocol=proto, n_tested=int(tt.sum()),
        MAE_single_all=np.abs(y - o["single"])[tt].mean(),
        MAE_two_stage_all=np.abs(y - o["two"])[tt].mean(),
        MAE_single_nonmetals=np.abs(y - o["single"])[m1].mean(),
        MAE_two_stage_nonmetals=np.abs(y - o["two"])[m1].mean(),
        MAE_regressor_if_nonmetal_known=np.abs(y - o["reg"])[m1].mean(),
        metal_vs_nonmetal_accuracy=((o["pnm"] > 0.5).astype(int) == nm)[tt].mean(),
        metal_vs_nonmetal_AUC=(roc_auc_score(nm[tt], o["pnm"][tt]) if len(set(nm[tt])) > 1 else np.nan),
        error90_nonmetals_eV=np.quantile(np.abs(y - o["reg"])[m1], 0.9)))
T = pd.DataFrame(rows).set_index("protocol").round(3)
T.to_csv(f"{OUT}/protocol_comparison.csv")
print(T.T.to_string())

# which anion families are hardest when held out entirely?
fo = OOF["family"]; fam_rows = []
for g in sorted(set(df.family.unique()) - {"other"}):
    mk = (df.family.values == g)
    mm = mk & (nm == 1)
    fam_rows.append(dict(held_out_family=g, n=int(mk.sum()), n_nonmetal=int(mm.sum()),
        MAE_regressor_nonmetals=float(np.abs(y - fo["reg"])[mm].mean()) if mm.sum() else np.nan,
        metal_vs_nonmetal_accuracy=float(((fo["pnm"][mk] > 0.5).astype(int) == nm[mk]).mean())))
FT = pd.DataFrame(fam_rows).round(3); FT.to_csv(f"{OUT}/family_holdout.csv", index=False)
print("\nHolding out a whole family (the 'other' family stays in training):\n", FT.to_string(index=False))

# residuals used later to turn predictions into probabilities (system protocol, true non-metals)
RES = (y - OOF["system"]["reg"])[nm == 1]
Q90 = float(np.quantile(np.abs(RES), 0.9))
print(f"\nTypical error for non-metals (system protocol): 90% of errors below {Q90:.2f} eV")

fig, axs = plt.subplots(1, 2, figsize=(8.4, 3.8))
o = OOF["system"]
axs[0].scatter(y, o["single"], s=3, alpha=.3); axs[0].set_title("Single-stage")
axs[1].scatter(y, o["two"], s=3, alpha=.3); axs[1].set_title("Two-stage")
for a in axs:
    lim = [0, 12]; a.plot(lim, lim, "k--", lw=.8); a.set_xlim(-.2, 12); a.set_ylim(-.2, 12)
    a.set_xlabel("Measured gap (eV)")
axs[0].set_ylabel("Predicted gap (eV)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_single_vs_two_stage.png", dpi=150); plt.show()

## Cell 4 — Define “low-cost”, then get REAL candidate compounds
Rule: abundance ≥ 2 ppm, not toxic, not scarce/costly. Abundance numbers are rounded textbook values — **verify against USGS/CRC before publishing**.
Candidates are Materials Project compounds (formation energy < 0 eV/atom, a *rough* stability filter — it is not the same as being stable against decomposition) made only of allowed elements, with at least one of O/S/N/P and no halogens. You can change the settings at the top of the cell.

In [ ]:
MIN_PPM = 2.0
ANION_KEEP = {"O", "S", "N", "P"}      # candidates must contain at least one of these
MAX_ELEMENTS = 4
USE_MP = True                           # set to False to use generated formulas instead (fallback)

ABUND_PPM = {"O":461000,"Si":282000,"Al":82300,"Fe":56300,"Ca":41500,"Na":23600,"Mg":23300,"K":20900,
 "Ti":5650,"P":1050,"Mn":950,"S":350,"Cl":145,"V":120,"Cr":102,"Zn":70,"Ni":84,"Cu":60,
 "Co":25,"Sn":2.3,"N":19,"Li":20,"Zr":165,"Ba":425,"Sr":370,"F":585,"Br":2.4,
 "Pb":14,"Cd":0.15,"As":1.8,"Se":0.05,"Te":0.001,"In":0.25,"Ga":19,"Ge":1.5,"Sb":0.2,"Bi":0.0085,"Ag":0.075}
TOXIC = {"Pb","Cd","As","Se","Hg","Tl","Be"}
SCARCE_OR_COSTLY = {"Te","In","Ga","Ge","Sb","Bi","Ag","Co"}
def allowed(el): return ABUND_PPM.get(el, 0) >= MIN_PPM and el not in TOXIC and el not in SCARCE_OR_COSTLY
print("Allowed elements:", sorted(e for e in ABUND_PPM if allowed(e)))

def load_mp_formulas():
    mp = load_dataset("matbench_mp_e_form")
    tcol = [c for c in mp.columns if pd.api.types.is_numeric_dtype(mp[c])][0]
    scol = [c for c in mp.columns if c != tcol][0]
    out = pd.DataFrame({"formula": [s.composition.reduced_formula for s in mp[scol]],
                        "e_form": mp[tcol].astype(float).values})
    del mp
    return out.groupby("formula", as_index=False).e_form.min()

def generate_formulas():
    OX = {"Li":[1],"Na":[1],"K":[1],"Mg":[2],"Ca":[2],"Sr":[2],"Ba":[2],"Al":[3],"Si":[4],"Ti":[4,3],"V":[5,3],
          "Cr":[3],"Mn":[2,4,3],"Fe":[2,3],"Ni":[2],"Cu":[1,2],"Zn":[2],"Sn":[2,4],"Zr":[4]}
    AN = {"O":2,"S":2,"N":3,"P":3}
    cats = [e for e in OX if allowed(e)]; ans = [a for a in AN if allowed(a) and a in ANION_KEEP]
    out = {}
    for ncat in (1, 2, 3):
        for cs in itertools.combinations(cats, ncat):
            oxo = list(itertools.product(*[OX[c] for c in cs]))
            for an in ans:
                for cc in itertools.product(range(1, 5), repeat=ncat):
                    for ac in range(1, 7):
                        if sum(cc) + ac > 10 or reduce(math.gcd, list(cc) + [ac]) != 1: continue
                        if any(sum(c*o for c, o in zip(cc, ox)) == AN[an]*ac for ox in oxo):
                            out["".join(f"{e}{c if c > 1 else ''}" for e, c in zip(cs, cc)) + f"{an}{ac if ac > 1 else ''}"] = np.nan
    return pd.DataFrame({"formula": list(out), "e_form": np.nan})

source = "Materials Project (matbench_mp_e_form)"
if USE_MP:
    try:
        print("Downloading Materials Project data (several minutes)...")
        pool = load_mp_formulas()
    except Exception as e:
        print("Could not load Materials Project data:", repr(e)[:200])
        print("Falling back to generated formulas.")
        pool = generate_formulas(); source = "generated charge-balanced formulas (fallback)"
else:
    pool = generate_formulas(); source = "generated charge-balanced formulas (fallback)"
print("Candidate source:", source, "| unique formulas:", len(pool))

pool["els"] = pool.formula.apply(lambda f: set(parse_formula(f)))
pool = pool[pool.els.apply(len) > 0]
pool = pool[pool.els.apply(lambda s: all(allowed(e) for e in s))]
pool = pool[pool.els.apply(lambda s: bool(s & ANION_KEEP) and not (s & HAL))]
pool = pool[pool.els.apply(len) <= MAX_ELEMENTS]
if pool.e_form.notna().any():
    pool = pool[pool.e_form < 0]
pool["key"] = pool.formula.apply(frac_key)
pool = pool[~pool.key.isin(set(df.key))].reset_index(drop=True)
pool["family"] = pool.formula.apply(family)
pool["n_elements"] = pool.els.apply(len)
pool["least_abundant_ppm"] = pool.els.apply(lambda s: min(ABUND_PPM[e] for e in s))
pool["system_in_training"] = pool.formula.apply(chem_system).isin(set(df.system))
print("Candidates after cost/toxicity/family/novelty filters:", len(pool))
print(pool.family.value_counts().to_dict())
assert len(pool) > 50, "Too few candidates - send me this output."

## Cell 5 — Predict gap and the probability of being in the 1.0–1.8 eV window

In [ ]:
print("Computing chemical features for candidates...")
Xc = featurize(pool.formula.tolist())

trn_all = np.where(nm == 1)[0]
REG = rf_reg(300).fit(X[trn_all], y[trn_all])
CLF = rf_clf(300).fit(X, nm)

pool["p_nonmetal"] = p_nonmetal(CLF, Xc)
pool["pred_gap_eV"] = REG.predict(Xc)
# probability in window: assume a candidate's error looks like out-of-fold errors on non-metals
inwin = ((pool.pred_gap_eV.values[:, None] + RES[None, :]) >= 1.0) & ((pool.pred_gap_eV.values[:, None] + RES[None, :]) <= 1.8)
pool["p_in_window"] = pool.p_nonmetal * inwin.mean(axis=1)
pool["band_lo_eV"] = (pool.pred_gap_eV - Q90).clip(lower=0)
pool["band_hi_eV"] = pool.pred_gap_eV + Q90
pool = pool.drop(columns=["els", "key"])
pool.sort_values("p_in_window", ascending=False).to_csv(f"{OUT}/all_candidates.csv", index=False)

print("Max probability of being in window: %.2f" % pool.p_in_window.max())
print("Candidates with p_in_window >= 0.30:", int((pool.p_in_window >= 0.30).sum()), "of", len(pool))
print("Candidates predicted non-metal (p>0.5):", int((pool.p_nonmetal > 0.5).sum()))

## Cell 6 — Honest test: rediscover known solar materials (their chemical system is hidden)

In [ ]:
KNOWN = {"Cu2ZnSnS4": 1.5, "Zn3P2": 1.5, "FeS2": 0.95, "SnS": 1.3, "Cu2O": 2.1, "BaZrS3": 1.8, "Fe2O3": 2.1}
rows = []
for f, lit in KNOWN.items():
    keep = (df.system.values != chem_system(f))
    xf = featurize([f])
    pn = float(p_nonmetal(rf_clf().fit(X[keep], nm[keep]), xf)[0])
    k2 = keep & (nm == 1)
    pr = float(rf_reg().fit(X[k2], y[k2]).predict(xf)[0])
    p1 = float(rf_reg().fit(X[keep], y[keep]).predict(xf)[0])
    rows.append(dict(compound=f, literature_gap_eV_approx=lit, single_stage=round(p1, 2),
                     p_nonmetal=round(pn, 2), two_stage_regressor=round(pr, 2),
                     err_single=round(abs(p1 - lit), 2), err_two_stage=round(abs(pr - lit), 2),
                     rows_removed=int((~keep).sum())))
rec = pd.DataFrame(rows); rec.to_csv(f"{OUT}/blind_recovery.csv", index=False)
print(rec.to_string(index=False))
print("\nMean error  single-stage: %.2f eV | two-stage: %.2f eV" % (rec.err_single.mean(), rec.err_two_stage.mean()))
print("Literature gaps are approximate and for orientation only.")

## Cell 7 — Top candidates, literature template, download

In [ ]:
cols = ["formula","family","n_elements","pred_gap_eV","p_nonmetal","p_in_window","band_lo_eV","band_hi_eV","e_form","least_abundant_ppm","system_in_training"]
top = pool.sort_values("p_in_window", ascending=False)
top.head(30)[cols].round(3).to_csv(f"{OUT}/top30_candidates.csv", index=False)
print("Top 30 by probability of being in the 1.0-1.8 eV window:")
print(top.head(30)[cols].round(2).to_string(index=False))
per_fam = pd.concat([top[top.family == g].head(8) for g in ["oxide","chalcogenide","pnictide","other"]])
per_fam[cols].round(3).to_csv(f"{OUT}/top8_per_family.csv", index=False)

fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.hist(pool.p_in_window, bins=30, color="#4C78A8")
ax.set_xlabel("Estimated probability of gap in 1.0-1.8 eV"); ax.set_ylabel("Candidates")
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_probability_hist.png", dpi=150); plt.show()

pd.DataFrame(columns=["formula","measured_gap_eV","phase_or_polymorph","gap_type(direct/indirect/optical)",
                      "method(Tauc UV-Vis/ellipsometry/PL)","temperature_K","doi","notes"]
             ).to_csv(f"{OUT}/literature_validation_TEMPLATE.csv", index=False)

summary = dict(n_dataset=int(len(df)), nonmetal_share=float(nm.mean()), n_systems=int(df.system.nunique()),
    protocol_comparison=T.reset_index().to_dict(orient="records"), family_holdout=FT.to_dict(orient="records"),
    error90_nonmetals_system_protocol=Q90, candidate_source=source, n_candidates=int(len(pool)),
    n_p_in_window_ge_0p30=int((pool.p_in_window >= 0.30).sum()), max_p_in_window=float(pool.p_in_window.max()),
    blind_recovery=rec.to_dict(orient="records"),
    mean_blind_err_single=float(rec.err_single.mean()), mean_blind_err_two_stage=float(rec.err_two_stage.mean()))
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2, default=float)
shutil.make_archive("step2_results", "zip", OUT)
print("\nSaved step2_results.zip")
try:
    from google.colab import files
    files.download("step2_results.zip")
except ImportError:
    pass

## When it finishes
Send me `summary.json` from `step2_results.zip` (or paste the printed output of Cells 3, 4, 6 and 7).

**Important:** candidates are model-based hypotheses. Negative formation energy is only a rough stability screen, and a good band gap does not make a working solar cell.